### Libraries and Depencies

In [ ]:
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
from pathlib import Path
import spacy
import numpy as np
import pandas as pd
import requests
import logging
import time
import re
import os
import json

import os
hf_token = os.getenv("HF_TOKEN")  # Hugging Face API token from environment variable

# Optional packages for assignment requirements
# pip install pinecone-client rank-bm25
try:
    from pinecone import Pinecone, ServerlessSpec
except Exception:
    Pinecone = None
    ServerlessSpec = None

try:
    from rank_bm25 import BM25Okapi
except Exception:
    BM25Okapi = None

### Chunking Strategy

In [2]:
# Arbitary Size Chunking
def Arbitary_chunking(text, chunk_size=512, overlap=64):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

In [3]:
# Recursive Chunking
def Recursive_chunking(text, target_size=800):
    sections = text.split('\n## ')
    chunks = []
    for section in sections:
        if len(section) > 1000:
            paragraphs = section.split('\n\n')
            current_chunk = ""
            for para in paragraphs:
                if len(current_chunk + para) < target_size:
                    current_chunk += para + "\n\n"
                else:
                    if current_chunk.strip():
                        chunks.append(current_chunk.strip())
                    current_chunk = para + "\n\n"
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
        else:
            if section.strip():
                chunks.append(section.strip())
    return chunks

In [4]:
# Semantic Chunking
def Semantic_chunking(text, max_chars=800):
    try:
        import spacy
        nlp = spacy.load('en_core_web_sm')
        doc = nlp(text)
        sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    except Exception:
        # Fallback if spaCy model is unavailable
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) < max_chars:
            current_chunk += sentence + " "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

def chunk_document(text, method='recursive'):
    if method == 'arbitary':
        return Arbitary_chunking(text)
    if method == 'semantic':
        return Semantic_chunking(text)
    return Recursive_chunking(text)

### Document Retrival

In [5]:
class HybridRetriever:
    """
    Hybrid retrieval with BM25 + Semantic (Pinecone) using RRF + optional re-ranking.
    """

    def __init__(self, index_name='rag-assignment3-index', embedding_model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.index_name = index_name
        self.pc_client = None
        self.pc_index = None

        self.bm25_model = None
        self.bm25_chunks = []

        self.intent_labels = ['factual query', 'procedural steps', 'conceptual explanation']
        self.classifier = pipeline(
            'zero-shot-classification',
            model='cross-encoder/nli-deberta-v3-small'
        )

        pinecone_api_key = os.getenv('PINECONE_API_KEY')
        pinecone_env = os.getenv('PINECONE_ENVIRONMENT', 'us-east-1')

        if Pinecone is not None and pinecone_api_key:
            self.pc_client = Pinecone(api_key=pinecone_api_key)
            existing_indexes = [idx.name for idx in self.pc_client.list_indexes()]
            if index_name not in existing_indexes:
                self.pc_client.create_index(
                    name=index_name,
                    dimension=384,
                    metric='cosine',
                    spec=ServerlessSpec(cloud='aws', region=pinecone_env)
                )
            self.pc_index = self.pc_client.Index(index_name)

    def classify_query(self, query: str) -> str:
        prediction = self.classifier(query, self.intent_labels)
        top_label = prediction['labels'][0].lower()
        if 'factual' in top_label:
            return 'factual'
        if 'procedural' in top_label:
            return 'procedural'
        return 'conceptual'

    def set_bm25_corpus(self, chunks: List[Dict[str, Any]]):
        self.bm25_chunks = chunks
        if BM25Okapi is None:
            raise ImportError('rank_bm25 is not installed. Run: pip install rank-bm25')
        tokenized = [c['text'].lower().split() for c in chunks]
        self.bm25_model = BM25Okapi(tokenized)

    def _bm25_search(self, query: str, top_k: int) -> List[Dict[str, Any]]:
        if self.bm25_model is None:
            return []
        tokenized_query = query.lower().split()
        scores = self.bm25_model.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []
        for idx in top_indices:
            item = dict(self.bm25_chunks[idx])
            item['id'] = item.get('id', f'bm25_{idx}')
            item['score'] = float(scores[idx])
            item['search_type'] = 'keyword'
            results.append(item)
        return results

    def _semantic_search(self, query: str, top_k: int) -> List[Dict[str, Any]]:
        if self.pc_index is None:
            return []
        query_vector = self.embedding_model.encode(query).tolist()
        response = self.pc_index.query(
            vector=query_vector,
            top_k=top_k,
            include_metadata=True
        )

        results = []
        for m in response.matches:
            meta = m.metadata or {}
            results.append({
                'id': m.id,
                'text': meta.get('text', ''),
                'source': meta.get('source', 'unknown'),
                'score': float(m.score),
                'search_type': 'semantic'
            })
        return results

    def _rrf_fusion(self, keyword_results, semantic_results, k=60):
        rrf_scores = {}
        merged_docs = {}

        for rank, doc in enumerate(keyword_results, start=1):
            doc_id = doc['id']
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            merged_docs[doc_id] = doc

        for rank, doc in enumerate(semantic_results, start=1):
            doc_id = doc['id']
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            if doc_id in merged_docs:
                merged_docs[doc_id]['search_type'] = 'hybrid'
            else:
                merged_docs[doc_id] = doc

        fused = []
        for doc_id, score in rrf_scores.items():
            doc = dict(merged_docs[doc_id])
            doc['rrf_score'] = score
            fused.append(doc)

        fused.sort(key=lambda x: x.get('rrf_score', 0.0), reverse=True)
        return fused

    def _rerank(self, query: str, results: List[Dict[str, Any]], top_k: int):
        if not results:
            return []
        query_vec = self.embedding_model.encode(query)
        reranked = []
        for doc in results:
            text_vec = self.embedding_model.encode(doc.get('text', ''))
            score = float(cosine_similarity([query_vec], [text_vec])[0][0])
            d = dict(doc)
            d['rerank_score'] = score
            reranked.append(d)
        reranked.sort(key=lambda x: x['rerank_score'], reverse=True)
        return reranked[:top_k]

    def retrieve_hybrid(self, query: str, top_k: int = 5, rerank: bool = True) -> List[Dict[str, Any]]:
        query_type = self.classify_query(query)
        if query_type == 'factual':
            keyword_results = self._bm25_search(query, top_k=8)
            semantic_results = self._semantic_search(query, top_k=4)
        else:
            keyword_results = self._bm25_search(query, top_k=4)
            semantic_results = self._semantic_search(query, top_k=8)

        fused = self._rrf_fusion(keyword_results, semantic_results)
        if rerank:
            return self._rerank(query, fused, top_k=top_k)
        return fused[:top_k]

    def retrieve_semantic_only(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        return self._semantic_search(query, top_k=top_k)

    def remove_duplicates(self, results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        seen_ids = set()
        unique_results = []
        for doc in results:
            doc_id = doc.get('id')
            if doc_id not in seen_ids:
                seen_ids.add(doc_id)
                unique_results.append(doc)
        return unique_results

In [6]:
utility_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embedding(text):
    return utility_embedding_model.encode(text)

def count_tokens(text):
    # Simple approximation for notebook experimentation
    return len(text.split())

def remove_duplicates(retrieved_chunks):
    seen = set()
    unique = []
    for chunk in retrieved_chunks:
        chunk_id = chunk.get('id', chunk.get('text', ''))
        if chunk_id not in seen:
            seen.add(chunk_id)
            unique.append(chunk)
    return unique

def adds_new_info(new_chunk, existing_chunks, similarity_threshold=0.8):
    """Check if new chunk provides information not in existing chunks"""
    if not existing_chunks:
        return True

    new_embedding = get_embedding(new_chunk['text'])
    for existing in existing_chunks:
        existing_embedding = get_embedding(existing['text'])
        sim = cosine_similarity([new_embedding], [existing_embedding])[0][0]
        if sim > similarity_threshold:
            return False
    return True

In [7]:
def intelligent_context_selection(query, retrieved_chunks, max_tokens=2000):
    # Remove duplicates and near-duplicates
    unique_chunks = remove_duplicates(retrieved_chunks)

    # Rank by complementarity - how well chunks work together
    selected_chunks = []
    total_tokens = 0

    for chunk in unique_chunks:
        if adds_new_info(chunk, selected_chunks):
            chunk_tokens = count_tokens(chunk['text'])
            if total_tokens + chunk_tokens <= max_tokens:
                selected_chunks.append(chunk)
                total_tokens += chunk_tokens

    return selected_chunks

In [8]:
def create_rag_prompt(query, context_chunks):
    context_text = "\n\n".join([
        f"Source {i+1}: {chunk['text']}"
        for i, chunk in enumerate(context_chunks)
    ])

    prompt = f"""CONTEXT:
{context_text}

QUESTION: {query}

INSTRUCTIONS:
1. Answer only from the provided context.
2. If information is missing, say clearly what is missing.
3. Keep answer concise and factual.
"""
    return prompt

def generate_answer_hf(prompt, hf_model='google/flan-t5-base'):
    from huggingface_hub import InferenceClient

    hf_token = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')
    candidate_models = [hf_model, 'google/flan-t5-large', 'bigscience/bloom-560m']
    last_error = None

    if hf_token and hf_token != 'your_hf_token_here':
        client = InferenceClient(provider='hf-inference', api_key=hf_token)
        for model_name in candidate_models:
            start = time.time()
            try:
                out = client.text_generation(
                    prompt,
                    model=model_name,
                    max_new_tokens=350,
                    temperature=0.2
                )
                latency = time.time() - start
                return out, latency
            except Exception as e:
                last_error = repr(e)
                continue

    # Local fallback when HF Inference is unavailable.
    global _local_gen_pipe
    if '_local_gen_pipe' not in globals() or _local_gen_pipe is None:
        _local_gen_pipe = pipeline('text-generation', model='distilgpt2')

    # Truncate prompt to fit local model context window
    local_prompt = prompt
    try:
        tok = _local_gen_pipe.tokenizer
        max_positions = int(getattr(_local_gen_pipe.model.config, 'n_positions', 1024))
        max_input_tokens = max(64, max_positions - 120)
        token_ids = tok.encode(prompt, add_special_tokens=False)
        if len(token_ids) > max_input_tokens:
            token_ids = token_ids[-max_input_tokens:]
            local_prompt = tok.decode(token_ids, skip_special_tokens=True)
    except Exception:
        pass

    start = time.time()
    local_out = _local_gen_pipe(local_prompt, max_new_tokens=120, do_sample=False)
    latency = time.time() - start
    if isinstance(local_out, list) and len(local_out) > 0:
        if 'generated_text' in local_out[0]:
            return local_out[0]['generated_text'], latency
        return str(local_out[0]), latency

    if last_error:
        return f'Fallback output unavailable. Remote error: {last_error}', latency
    return str(local_out), latency

### Logging

In [9]:
class RAGLogger:
    def __init__(self):
        self.logger = logging.getLogger('rag_system')
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
            self.logger.addHandler(handler)

    def log_rag_interaction(self, query, retrieved_chunks, response, metrics=None, user_feedback=None):
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'query': query,
            'retrieved_chunks': [
                {
                    'id': chunk.get('id', 'na'),
                    'score': chunk.get('score', chunk.get('rerank_score', 0.0)),
                    'source': chunk.get('source', 'unknown')
                } for chunk in retrieved_chunks
            ],
            'response': response,
            'metrics': metrics or {},
            'user_feedback': user_feedback,
            'retrieval_count': len(retrieved_chunks)
        }
        self.logger.info(f'RAG_INTERACTION: {log_entry}')

    def analyze_failures(self, time_period='7d'):
        # Extend later by querying persisted logs
        pass

### Corpus Loading and Indexing

In [10]:
# TODO: Add your corpus folder path here later
CORPUS_PATH = r'C:\\Users\\khatw\\OneDrive\\Desktop\\IBA\\acedemics\\Semester_6\\NLP\\RAG\\synthetic_knowledge_items.csv'

CHUNKING_METHOD = 'recursive'   # 'arbitary' | 'recursive' | 'semantic'
TOP_K = 5

def read_corpus_documents(corpus_path):
    corpus_path = Path(corpus_path)
    docs = []

    # Helper to load a plain text/markdown file
    def _load_text_file(fp):
        text = fp.read_text(encoding='utf-8', errors='ignore').strip()
        if text:
            docs.append({'source': str(fp), 'text': text})

    # Helper to load a CSV file into one document per row
    def _load_csv_file(fp):
        df = pd.read_csv(fp)
        text_cols = [c for c in df.columns if df[c].dtype == 'object']
        if len(text_cols) == 0:
            text_cols = list(df.columns)

        for idx, row in df.iterrows():
            row_parts = []
            for col in text_cols:
                val = row.get(col, None)
                if pd.notna(val):
                    s = str(val).strip()
                    if s:
                        row_parts.append(f'{col}: {s}')
            if row_parts:
                docs.append({
                    'source': f'{fp}#row={idx}',
                    'text': '\n'.join(row_parts)
                })

    if corpus_path.is_file():
        suffix = corpus_path.suffix.lower()
        if suffix in ['.txt', '.md']:
            _load_text_file(corpus_path)
        elif suffix == '.csv':
            _load_csv_file(corpus_path)
        return docs

    # If a directory is provided, scan supported file types recursively
    for pattern in ['*.txt', '*.md']:
        for fp in corpus_path.rglob(pattern):
            _load_text_file(fp)

    for fp in corpus_path.rglob('*.csv'):
        _load_csv_file(fp)

    return docs

In [11]:
def build_chunks_from_docs(docs, method='recursive'):
    chunks = []
    chunk_counter = 0
    for doc in docs:
        local_chunks = chunk_document(doc['text'], method=method)
        for ch in local_chunks:
            chunks.append({
                'id': f'ch_{chunk_counter}',
                'text': ch,
                'source': doc['source']
            })
            chunk_counter += 1
    return chunks

def upsert_chunks_to_pinecone(retriever: HybridRetriever, chunks, batch_size=100):
    if retriever.pc_index is None:
        print('Pinecone not configured. Set PINECONE_API_KEY and retry.')
        return

    vectors = []
    for chunk in chunks:
        vec = retriever.embedding_model.encode(chunk['text']).tolist()
        vectors.append({
            'id': chunk['id'],
            'values': vec,
            'metadata': {'text': chunk['text'], 'source': chunk['source']}
        })

    for i in range(0, len(vectors), batch_size):
        retriever.pc_index.upsert(vectors=vectors[i:i + batch_size])

def prepare_retrieval_indexes(corpus_path, chunking_method='recursive'):
    docs = read_corpus_documents(corpus_path)
    if len(docs) == 0:
        raise ValueError('No documents found. Add files in your corpus path and rerun.')

    chunks = build_chunks_from_docs(docs, method=chunking_method)
    retriever = HybridRetriever()
    retriever.set_bm25_corpus(chunks)
    upsert_chunks_to_pinecone(retriever, chunks)

    print(f'Documents loaded: {len(docs)}')
    print(f'Chunks created: {len(chunks)}')
    return retriever, docs, chunks

### LLM-as-a-Judge Evaluation (Faithfulness + Relevancy)

In [12]:
def call_hf_judge(prompt, model='google/flan-t5-base'):
    from huggingface_hub import InferenceClient

    hf_token = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')
    candidate_models = [model, 'google/flan-t5-large', 'bigscience/bloom-560m']
    last_error = None

    if hf_token and hf_token != 'your_hf_token_here':
        client = InferenceClient(provider='hf-inference', api_key=hf_token)
        for model_name in candidate_models:
            try:
                out = client.text_generation(
                    prompt,
                    model=model_name,
                    max_new_tokens=256,
                    temperature=0.0
                )
                return out
            except Exception as e:
                last_error = repr(e)
                continue

    # Local fallback when HF Inference is unavailable.
    global _local_judge_pipe
    if '_local_judge_pipe' not in globals() or _local_judge_pipe is None:
        _local_judge_pipe = pipeline('text-generation', model='distilgpt2')

    # Truncate prompt to fit local model context window
    local_prompt = prompt
    try:
        tok = _local_judge_pipe.tokenizer
        max_positions = int(getattr(_local_judge_pipe.model.config, 'n_positions', 1024))
        max_input_tokens = max(64, max_positions - 80)
        token_ids = tok.encode(prompt, add_special_tokens=False)
        if len(token_ids) > max_input_tokens:
            token_ids = token_ids[-max_input_tokens:]
            local_prompt = tok.decode(token_ids, skip_special_tokens=True)
    except Exception:
        pass

    local_out = _local_judge_pipe(local_prompt, max_new_tokens=80, do_sample=False)
    if isinstance(local_out, list) and len(local_out) > 0:
        if 'generated_text' in local_out[0]:
            return local_out[0]['generated_text']
        return str(local_out[0])

    if last_error:
        return f'Fallback output unavailable. Remote error: {last_error}'
    return str(local_out)

def extract_claims(answer_text):
    prompt = f"""Extract atomic factual claims from the answer.
Return only a JSON array of short claims.
Answer: {answer_text}"""
    out = call_hf_judge(prompt)
    try:
        arr_match = re.search(r'\[.*\]', out, re.DOTALL)
        if arr_match:
            return list(pd.json_normalize(eval(arr_match.group(0))).values.flatten())
    except Exception:
        pass
    claims = [line.strip('- ').strip() for line in out.split('\n') if line.strip()]
    return [c for c in claims if len(c) > 5][:8]

def verify_claims_against_context(claims, context_text):
    verdicts = []
    for claim in claims:
        prompt = f"""Context:\n{context_text}\n\nClaim: {claim}\n\nIs this claim supported by context? Reply only with SUPPORTED or UNSUPPORTED."""
        out = call_hf_judge(prompt).upper()
        supported = 'SUPPORTED' in out and 'UNSUPPORTED' not in out
        verdicts.append({'claim': claim, 'supported': supported})
    return verdicts

def faithfulness_score(answer_text, retrieved_chunks):
    context_text = '\n\n'.join([c['text'] for c in retrieved_chunks])
    claims = extract_claims(answer_text)
    if len(claims) == 0:
        return 0.0, []
    verdicts = verify_claims_against_context(claims, context_text)
    score = sum(v['supported'] for v in verdicts) / len(verdicts)
    return score, verdicts

def generate_alternate_queries(answer_text):
    prompt = f"""Generate 3 alternative user questions that would have answer below.
Return only one question per line.\n\nAnswer:\n{answer_text}"""
    out = call_hf_judge(prompt)
    lines = [l.strip(' -').strip() for l in out.split('\n') if l.strip()]
    return lines[:3]

def relevancy_score(original_query, answer_text, embedding_model=None):
    if embedding_model is None:
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    alt_qs = generate_alternate_queries(answer_text)
    if len(alt_qs) == 0:
        return 0.0, []

    q_vec = embedding_model.encode(original_query)
    sims = []
    for q in alt_qs:
        q2_vec = embedding_model.encode(q)
        sims.append(float(cosine_similarity([q_vec], [q2_vec])[0][0]))

    return float(np.mean(sims)), sims

In [13]:
def run_single_query_pipeline(query, retriever, retrieval_mode='hybrid', top_k=5):
    t0 = time.time()
    if retrieval_mode == 'semantic':
        retrieved = retriever.retrieve_semantic_only(query, top_k=top_k)
    else:
        retrieved = retriever.retrieve_hybrid(query, top_k=top_k, rerank=True)
    retrieval_latency = time.time() - t0

    selected_chunks = intelligent_context_selection(query, retrieved, max_tokens=2000)
    prompt = create_rag_prompt(query, selected_chunks)

    answer, generation_latency = generate_answer_hf(prompt)
    faith_score, claim_verification = faithfulness_score(answer, selected_chunks)
    rel_score, rel_sims = relevancy_score(query, answer, retriever.embedding_model)

    total_latency = retrieval_latency + generation_latency
    metrics = {
        'faithfulness': faith_score,
        'relevancy': rel_score,
        'retrieval_latency': retrieval_latency,
        'generation_latency': generation_latency,
        'total_latency': total_latency
    }

    return {
        'query': query,
        'answer': answer,
        'retrieved_chunks': selected_chunks,
        'claim_verification': claim_verification,
        'relevancy_similarities': rel_sims,
        'metrics': metrics
    }

### Ablation Study (Chunking + Retrieval)

In [14]:
def run_ablation(test_queries, corpus_path):
    settings = [
        {'chunking': 'arbitary', 'retrieval': 'semantic'},
        {'chunking': 'recursive', 'retrieval': 'semantic'},
        {'chunking': 'recursive', 'retrieval': 'hybrid'},
        {'chunking': 'semantic', 'retrieval': 'hybrid'}
    ]

    rows = []
    for cfg in settings:
        retriever, _, _ = prepare_retrieval_indexes(corpus_path, chunking_method=cfg['chunking'])

        faithfulness_vals = []
        relevancy_vals = []
        latency_vals = []

        for q in test_queries:
            out = run_single_query_pipeline(
                query=q,
                retriever=retriever,
                retrieval_mode=cfg['retrieval'],
                top_k=TOP_K
            )
            faithfulness_vals.append(out['metrics']['faithfulness'])
            relevancy_vals.append(out['metrics']['relevancy'])
            latency_vals.append(out['metrics']['total_latency'])

        rows.append({
            'chunking': cfg['chunking'],
            'retrieval': cfg['retrieval'],
            'faithfulness_avg': float(np.mean(faithfulness_vals)),
            'relevancy_avg': float(np.mean(relevancy_vals)),
            'latency_avg_sec': float(np.mean(latency_vals))
        })

    return pd.DataFrame(rows).sort_values(['faithfulness_avg', 'relevancy_avg'], ascending=False)

In [19]:
# Example fixed test queries for assignment evaluation (10-20 recommended)
TEST_QUERIES = [
    'What are the exact steps to configure VPN access for remote workers?',
    'How do I set up a mobile device for company email, and what are the prerequisites?',
    'What is the procedure for setting up a new user account in JIRA?',
    "How do I assign security roles when setting up a new user's account in Oracle?",
    'What should I do if my Microsoft Office applications keep crashing or freezing?',
    'How can I reset my PIN if I have forgotten it?',
    'What information is required to create a new IT Incident or problem report?',
    'I am an IT specialist; what details do I need to gather before creating an IT request for a new employee?',

]

# Run these after setting CORPUS_PATH and API keys
retriever, docs, chunks = prepare_retrieval_indexes(CORPUS_PATH, chunking_method=CHUNKING_METHOD)
sample_output = run_single_query_pipeline(TEST_QUERIES[0], retriever, retrieval_mode='hybrid', top_k=TOP_K)

print('\n=== SAMPLE QUERY ===')
print(sample_output['query'])
print('\n=== GENERATED ANSWER ===')
print(sample_output['answer'])
print('\n=== METRICS ===')
print(sample_output['metrics'])
print('\n=== TOP CONTEXT CHUNKS (preview) ===')
for i, c in enumerate(sample_output['retrieved_chunks'][:3], start=1):
    print(f'[{i}]', c.get('text', '')[:250], '...')

ablation_df = run_ablation(TEST_QUERIES, CORPUS_PATH)
ablation_df

Device set to use cpu


Pinecone not configured. Set PINECONE_API_KEY and retry.
Documents loaded: 100
Chunks created: 1272


c:\Users\khatw\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\khatw\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=i


=== SAMPLE QUERY ===
What are the exact steps to configure VPN access for remote workers?

=== GENERATED ANSWER ===
CONTEXT:
Source 1: By following these steps, remote workers can securely connect to the company network and access company resources from anywhere. If you have any questions or issues, please contact the IT helpdesk for further assistance.
alt_ki_text: To configure VPN access for remote workers at Widgetco, follow these steps:

**Prerequisites:**

* Ensure the remote worker has a valid Widgetco username and password.
* Verify the remote worker's device meets the minimum system requirements for VPN access (Windows 10 or macOS High Sierra or later, with a compatible browser).
* Have the remote worker's device connected to a stable internet connection.

**Step 1: Install the VPN Client**

Source 2: * If required, have the remote worker configure their VPN settings to use a specific VPN server or protocol.
* Instruct them to consult the Widgetco VPN configuration guide ([htt

Device set to use cpu


...

Pinecone not configured. Set PINECONE_API_KEY and retry.
Documents loaded: 100
Chunks created: 1272


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Pinecone not configured. Set PINECONE_API_KEY and retry.
Documents loaded: 100
Chunks created: 1141


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

,chunking,retrieval,faithfulness_avg,relevancy_avg,latency_avg_sec
3,semantic,hybrid,0.0,0.115813,9.271963
0,arbitary,semantic,0.0,0.046615,6.307198
1,recursive,semantic,0.0,0.046615,6.173047
2,recursive,hybrid,0.0,0.046615,9.262007


### Live Web Interface (Streamlit Export Template)

In [20]:
APP_TEMPLATE = '''
import os
import streamlit as st

st.set_page_config(page_title='RAG Assignment 3', layout='wide')
st.title('RAG-based Question Answering System')

st.markdown('Set environment variables: HF_TOKEN, PINECONE_API_KEY, PINECONE_ENVIRONMENT')
st.markdown('Use the same retrieval and evaluation functions from your notebook in app.py.')

query = st.text_input('Ask a question:')

if query:
    st.info('Connect this app with your notebook pipeline functions.')
    st.write('Generated Answer: ...')
    st.write('Retrieved Context: ...')
    st.write('Faithfulness Score: ...')
    st.write('Relevancy Score: ...')
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_TEMPLATE)

print('app.py template written. Fill in pipeline calls and deploy to Hugging Face Spaces.')

app.py template written. Fill in pipeline calls and deploy to Hugging Face Spaces.


In [21]:
# Quick smoke test (single query) before full ablation
retriever, docs, chunks = prepare_retrieval_indexes(CORPUS_PATH, chunking_method=CHUNKING_METHOD)
smoke_output = run_single_query_pipeline(TEST_QUERIES[0], retriever, retrieval_mode='hybrid', top_k=TOP_K)
smoke_output['metrics']

Device set to use cpu


Pinecone not configured. Set PINECONE_API_KEY and retry.
Documents loaded: 100
Chunks created: 1272


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'faithfulness': 0.0,
 'relevancy': 0.011857213142017523,
 'retrieval_latency': 2.9798972606658936,
 'generation_latency': 12.066181898117065,
 'total_latency': 15.046079158782959}

In [22]:
# HF connectivity sanity check
from huggingface_hub import HfApi
hf_token_check = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')
print('Token present:', bool(hf_token_check))
if hf_token_check:
    try:
        who = HfApi().whoami(token=hf_token_check)
        print('Authenticated as:', who.get('name', 'unknown'))
    except Exception as e:
        print('HF whoami failed:', repr(e))

Token present: True
Authenticated as: Ibrahim1rfan


In [23]:
# HF generation probe
from huggingface_hub import InferenceClient
probe_client = InferenceClient(token=(os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')))
for probe_model in ['google/flan-t5-base', 'google/flan-t5-large', 'bigscience/bloom-560m']:
    try:
        probe_out = probe_client.text_generation('Hello', model=probe_model, max_new_tokens=10)
        print(probe_model, 'OK', probe_out)
    except Exception as e:
        print(probe_model, 'FAIL', repr(e))

google/flan-t5-base FAIL StopIteration()
google/flan-t5-large FAIL StopIteration()
bigscience/bloom-560m FAIL StopIteration()
